# ISS projekt 2025/26 — Music search (řešení)
**Autor:** *doplnit jméno*  
**Login:** *doplnit login*  
**Datum:** 2025-11-05

Notebook řeší úlohu Music search dvěma metodami: (A) Shazam-like fingerprinty, (B) MFCC embeddingy + kNN.
Je připraven pro běh nad složkami `data/known`, `data/valid`, `data/login` (mono, 16 kHz).

In [ ]:

# === Konfigurace cest ===
from pathlib import Path

DATA_DIR = Path("./data")
KNOWN_DIR = DATA_DIR / "known"
VALID_DIR = DATA_DIR / "valid"
LOGIN_DIR = DATA_DIR / "login"

import os, math, json, pickle, itertools, statistics, random
import numpy as np
import scipy.signal as sps
import scipy.fftpack as sfft
import soundfile as sf
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

FS = 16_000
RESULTS_DIR = Path("./results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("./cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())


In [ ]:

# === Pomocné funkce ===
def load_wav(path: Path, target_fs=FS):
    x, fs = sf.read(str(path), always_2d=False)
    if isinstance(x, np.ndarray) and x.ndim > 1:
        x = np.mean(x, axis=1)
    if fs != target_fs:
        import math
        gcd = math.gcd(fs, target_fs)
        up = target_fs // gcd
        down = fs // gcd
        x = sps.resample_poly(x, up, down)
    return np.asarray(x, dtype=np.float32)

def stft_mag(x, n_fft=1024, hop=256, win="hann"):
    w = sps.get_window(win, n_fft, fftbins=True)
    _, _, Z = sps.stft(x, nperseg=n_fft, noverlap=n_fft-hop, window=w, boundary=None)
    return np.abs(Z).T  # t x f


In [ ]:

# === Metoda A (Shazam-like) ===
def find_peaks(spec, neighborhood=(9, 9), amp_min_db=-60.0):
    S = 20*np.log10(np.maximum(spec, 1e-12))
    from scipy.ndimage import maximum_filter
    max_filt = maximum_filter(S, size=neighborhood, mode='reflect')
    mask = (S == max_filt) & (S > amp_min_db)
    return np.argwhere(mask)  # (t,f)

def build_hashes_from_peaks(peaks, fan_out=5, t_delta_min=1, t_delta_max=200, f_delta_max=200):
    hashes = []
    if len(peaks) == 0: return hashes
    peaks = peaks[np.argsort(peaks[:,0])]
    for i in range(len(peaks)):
        t1, f1 = peaks[i]
        for j in range(1, fan_out+1):
            if i+j >= len(peaks): break
            t2, f2 = peaks[i+j]
            dt = t2 - t1
            if dt < t_delta_min or dt > t_delta_max: continue
            if abs(f2 - f1) > f_delta_max: continue
            hashes.append(((int(f1), int(f2), int(dt)), int(t1)))
    return hashes

def index_known_shazam(known_paths, n_fft=1024, hop=256, peak_neigh=(9,9), amp_min_db=-55.0,
                       fan_out=8, t_delta_min=1, t_delta_max=200, f_delta_max=200):
    index = {}; song_meta = []
    for sid, p in enumerate(tqdm(known_paths, desc="Indexing known (Shazam)")):
        x = load_wav(p)
        spec = stft_mag(x, n_fft=n_fft, hop=hop)
        peaks = find_peaks(spec, neighborhood=peak_neigh, amp_min_db=amp_min_db)
        hashes = build_hashes_from_peaks(peaks, fan_out=fan_out, t_delta_min=t_delta_min,
                                         t_delta_max=t_delta_max, f_delta_max=f_delta_max)
        for key, t1 in hashes:
            index.setdefault(key, []).append((sid, t1))
        song_meta.append({"song_id": sid, "path": str(p), "n_frames": spec.shape[0], "n_samples": len(x)})
    return index, song_meta

def query_shazam(xq, index, song_meta, n_fft=1024, hop=256, peak_neigh=(9,9), amp_min_db=-55.0,
                 fan_out=8, t_delta_min=1, t_delta_max=200, f_delta_max=200, topk=5):
    spec = stft_mag(xq, n_fft=n_fft, hop=hop)
    peaks = find_peaks(spec, neighborhood=peak_neigh, amp_min_db=amp_min_db)
    hashes = build_hashes_from_peaks(peaks, fan_out=fan_out, t_delta_min=t_delta_min,
                                     t_delta_max=t_delta_max, f_delta_max=f_delta_max)
    votes = {}
    for key, t1_q in hashes:
        if key not in index: continue
        for sid, t1_db in index[key]:
            offset = t1_db - t1_q
            votes[(sid, offset)] = votes.get((sid, offset), 0) + 1
    song_scores = {}
    for (sid, offset), v in votes.items():
        song_scores[sid] = max(song_scores.get(sid, 0), v)
    ranked = sorted(song_scores.items(), key=lambda kv: kv[1], reverse=True)
    return ranked[:topk], votes


In [ ]:

# === Metoda B (MFCC + kNN) ===
def mel_filterbank(n_fft=1024, fs=16_000, n_mels=40, fmin=0, fmax=None):
    if fmax is None: fmax = fs/2
    def hz_to_mel(f): return 2595*np.log10(1+f/700)
    def mel_to_hz(m): return 700*(10**(m/2595)-1)
    mel_edges = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels+2)
    hz_edges = mel_to_hz(mel_edges)
    bins = np.floor((n_fft//2+1)*hz_edges/(fs/2)).astype(int)
    fb = np.zeros((n_mels, n_fft//2+1), dtype=np.float32)
    for m in range(1, n_mels+1):
        f_left, f_center, f_right = bins[m-1], bins[m], bins[m+1]
        if f_center == f_left: f_center += 1
        for k in range(f_left, f_center):
            fb[m-1, k] = (k - f_left) / max(1, (f_center - f_left))
        for k in range(f_center, f_right):
            fb[m-1, k] = (f_right - k) / max(1, (f_right - f_center))
    return fb

def compute_mfcc(x, n_fft=1024, hop=256, n_mels=40, n_ceps=20, lifter=22):
    spec = stft_mag(x, n_fft=n_fft, hop=hop)  # t x f
    fb = mel_filterbank(n_fft=n_fft, fs=FS, n_mels=n_mels)
    pow_spec = (spec ** 2)
    mel_spec = np.dot(pow_spec, fb.T)  # t x n_mels
    mel_spec = np.maximum(mel_spec, 1e-10)
    log_mel = np.log(mel_spec)
    ceps = sfft.dct(log_mel, type=2, n=n_ceps, axis=1, norm='ortho')
    if lifter:
        n = np.arange(n_ceps)
        lift = 1 + (lifter / 2.0) * np.sin(np.pi * n / lifter)
        ceps = ceps * lift
    return ceps  # t x n_ceps

def emb_from_mfcc(mfcc):
    d = np.diff(mfcc, axis=0, prepend=mfcc[:1])
    feats = np.concatenate([mfcc.mean(axis=0), mfcc.std(axis=0),
                            d.mean(axis=0), d.std(axis=0)], axis=0)
    return feats.astype(np.float32)

def index_known_mfcc(known_paths, n_fft=1024, hop=256, n_mels=40, n_ceps=20, pca_dim=48):
    X = []; meta = []
    for sid, p in enumerate(tqdm(known_paths, desc="Indexing known (MFCC)")):
        x = load_wav(p)
        mf = compute_mfcc(x, n_fft=n_fft, hop=hop, n_mels=n_mels, n_ceps=n_ceps)
        emb = emb_from_mfcc(mf)
        X.append(emb); meta.append({"song_id": sid, "path": str(p)})
    X = np.vstack(X) if len(X) else np.zeros((0, n_ceps*4), dtype=np.float32)
    pca = None
    if X.size and pca_dim and X.shape[1] > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=42)
        X = pca.fit_transform(X)
    knn = NearestNeighbors(n_neighbors=10, metric="cosine").fit(X) if X.size else None
    return {"knn": knn, "pca": pca, "meta": meta}

def query_mfcc(xq, index, n_fft=1024, hop=256, n_mels=40, n_ceps=20, topk=5):
    mf = compute_mfcc(xq, n_fft=n_fft, hop=hop, n_mels=n_mels, n_ceps=n_ceps)
    emb = emb_from_mfcc(mf)[None,:]
    if index["pca"] is not None:
        emb = index["pca"].transform(emb)
    if index["knn"] is None or len(index["meta"]) == 0:
        return []
    dists, idxs = index["knn"].kneighbors(emb, n_neighbors=min(topk, len(index["meta"])), return_distance=True)
    ranked = [(int(index["meta"][i]["song_id"]), float(1-d)) for i, d in zip(idxs[0], dists[0])]
    return ranked


In [ ]:

# === Data a indexace ===
def list_wavs_in_dir(d: Path):
    return sorted([p for p in d.glob("*.wav")], key=lambda p: int(p.stem))

def prepare_known():
    assert (Path("./data")/"known").exists(), "Chybí složka data/known"
    return list_wavs_in_dir(Path("./data")/"known")

known_dir = Path("./data")/"known"
known_paths = list_wavs_in_dir(known_dir) if known_dir.exists() else []
print("Nalezeno známých nahrávek:", len(known_paths))

if known_paths:
    shz_index, shz_meta = index_known_shazam(known_paths)
    import pickle, os
    os.makedirs("./cache", exist_ok=True)
    pickle.dump((shz_index, shz_meta), open("./cache/shazam_index.pkl", "wb"))
    mfcc_index = index_known_mfcc(known_paths)
    pickle.dump(mfcc_index, open("./cache/mfcc_index.pkl", "wb"))
    print("Indexy uloženy do ./cache")
else:
    print("Nebyla nalezena data, indexy se nevytvořily.")


In [ ]:

# === Vyhodnocení na validační sadě ===
from pathlib import Path
def load_keys(valid_dir: Path):
    key_file = valid_dir / "key.txt"
    pairs = []
    with open(key_file, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln: continue
            q, k = ln.split()
            pairs.append((int(q), int(k)))
    return pairs

def evaluate_valid(method="shazam", topk=5):
    valid_dir = Path("./data")/"valid"
    assert valid_dir.exists(), "Chybí složka data/valid"
    queries = list_wavs_in_dir(valid_dir)
    keys = {q: k for q, k in load_keys(valid_dir)}
    top5_hits = 0; top1_hits = 0

    import pickle
    if method == "shazam":
        shz_index, shz_meta = pickle.load(open("./cache/shazam_index.pkl", "rb"))
    else:
        mfcc_index = pickle.load(open("./cache/mfcc_index.pkl", "rb"))

    for qp in tqdm(queries, desc=f"Eval {method}"):
        xq = load_wav(qp)
        if method == "shazam":
            ranked, _ = query_shazam(xq, shz_index, shz_meta, topk=topk)
            ranked_ids = [sid for sid,_ in ranked]
        else:
            ranked = query_mfcc(xq, mfcc_index, topk=topk)
            ranked_ids = [sid for sid,_ in ranked]

        qid = int(qp.stem); gt = keys[qid]
        if len(ranked_ids) and ranked_ids[0] == gt: top1_hits += 1
        if gt in ranked_ids: top5_hits += 1

    n = max(1, len(queries))
    acc1 = top1_hits / n; acc5 = top5_hits / n
    print(f"{method} — 1-best acc: {acc1:.3f} | 5-best acc: {acc5:.3f}")
    return {"acc1": acc1, "acc5": acc5}

valid_dir = Path("./data")/"valid"
if (Path("./data")/"known").exists() and valid_dir.exists():
    res_shz = evaluate_valid("shazam", topk=5)
    res_mfc = evaluate_valid("mfcc", topk=5)
    import json, os
    os.makedirs("./results", exist_ok=True)
    json.dump({"shazam": res_shz, "mfcc": res_mfc}, open("./results/valid_results.json","w"), indent=2)
else:
    print("Pro vyhodnocení chybí data.")


In [ ]:

# === Predikce pro eval sadu (login) ===
from pathlib import Path
def predict_login(method="shazam", topk=5, out_file="./results/login_pred.txt"):
    login_dir = Path("./data")/"login"
    assert login_dir.exists(), "Chybí složka data/login"
    queries = list_wavs_in_dir(login_dir)
    import pickle, os
    if method == "shazam":
        shz_index, shz_meta = pickle.load(open("./cache/shazam_index.pkl", "rb"))
    else:
        mfcc_index = pickle.load(open("./cache/mfcc_index.pkl", "rb"))
    lines = []
    for qp in tqdm(queries, desc=f"Predict {method}"):
        xq = load_wav(qp)
        if method == "shazam":
            ranked, _ = query_shazam(xq, shz_index, shz_meta, topk=topk)
            pred = ranked[0][0] if ranked else -1
        else:
            ranked = query_mfcc(xq, mfcc_index, topk=topk)
            pred = ranked[0][0] if ranked else -1
        lines.append(f"{int(qp.stem)} {pred}")
    Path(out_file).write_text("\n".join(lines), encoding="utf-8")
    print("Uloženo:", Path(out_file).resolve())
    return out_file

# Automaticky vytvořit oba soubory, pokud data existují
if (Path("./data")/"known").exists() and (Path("./data")/"login").exists():
    predict_login("shazam", out_file="./results/login_pred.txt")
    predict_login("mfcc", out_file="./results/login_pred_mfcc.txt")
else:
    print("Predikce neproběhla (chybí data).")


In [ ]:
from pathlib import Path
Path('./results').mkdir(parents=True, exist_ok=True)
open('./results/protokol.md','w',encoding='utf-8').write("# Protokol \u2014 ISS Music search (\u0159e\u0161en\u00ed)\n\n**Autor:** *doplnit jm\u00e9no*  \n**Login:** *doplnit login*  \n**Datum:** 2025-11-05\n\n## Zad\u00e1n\u00ed\nVyhled\u00e1v\u00e1n\u00ed kr\u00e1tk\u00fdch dotazov\u00fdch audio-klip\u016f ve v\u011bt\u0161\u00ed datab\u00e1zi zn\u00e1m\u00fdch nahr\u00e1vek (mono, 16 kHz). Metriky: 1-best a 5-best p\u0159esnost na valida\u010dn\u00ed sad\u011b. Odevzd\u00e1n\u00ed obsahuje protokol a predikce pro evalua\u010dn\u00ed sadu (login).\n\n## Pr\u016fzkum vhodn\u00fdch p\u0159\u00edstup\u016f\n- Fingerprinty (Shazam-like): robustn\u00ed k hluku a posunu v \u010dase, rychl\u00e9 vyhled\u00e1v\u00e1n\u00ed p\u0159es hashe peak\u2011pair\u016f.\n- MFCC embeddingy + kNN: glob\u00e1ln\u00ed timbre podobnost, jednoduch\u00e1 implementace a lad\u011bn\u00ed, men\u0161\u00ed pam\u011b\u0165.\n- Roz\u0161\u00ed\u0159en\u00ed (neimplementov\u00e1no): log-mel CNN embeddingy, p\u0159\u00edpadn\u011b chroma/tempo popisy.\n\n## Implementace\n- Metoda A (Shazam): log-spektrum -> detekce vrchol\u016f (max-filter), tvorba p\u00e1r\u016f (fan-out), hash a hlasov\u00e1n\u00ed pro (song_id, offset).\n- Metoda B (MFCC): MFCC z log-melu (DCT-II), agregace [mean | std | delta-mean | delta-std], PCA (voliteln\u011b), kNN (kosinov\u00e1).\n\n## V\u00fdsledky na valida\u010dn\u00ed sad\u011b\nPo spu\u0161t\u011bn\u00ed \u010d\u00e1sti *Vyhodnocen\u00ed na valida\u010dn\u00ed sad\u011b* se dopln\u00ed do `results/valid_results.json`.\n\n## Diskuse a lad\u011bn\u00ed\n- Shazam: prahy (dB), neighborhood, fan-out, rozsahy \u0394t a \u0394f, n_fft/hop.\n- MFCC: n_mels, n_ceps, liftering, \u0394/\u0394\u0394, PCA dimenze, volba metriky.\n\n## Z\u00e1v\u011br\nMetody se dob\u0159e dopl\u0148uj\u00ed: fingerprinty obvykle d\u00e1vaj\u00ed vy\u0161\u0161\u00ed 1-best p\u0159esnost; MFCC mohou b\u00fdt u\u017eite\u010dn\u00e9 jako z\u00e1loha.\n")
print('Vytvořen soubor protokolu: ./results/protokol.md')
